# Project 15 — Evaluation Harness (exact-match, BLEU/ROUGE, classification, LLM-as-judge)

```
two-word reversal, COMPOSITIONAL split (novel pairs of known words)
  -> 3 live systems: base (frozen, zero-shot) vs LoRA-tuned (knowledge in weights)
     vs RAG (lexical retrieval + compose-the-facts reader, knowledge in context)
  -> one harness: EM, BLEU, ROUGE-1/L, sklearn classification metrics, rubric judge
  -> system comparison + RAG top-k sweep (CSVs + PNG)
```

Goal: stop eyeballing generations — score them like a benchmark, and split
retrieval quality from generation quality.

Learning objectives:
1. Implement BLEU (clipped precision + brevity penalty) and ROUGE-1/L (F1) from scratch.
2. Use sklearn for classification-style scoring over predicted words.
3. Write a rubric LLM-as-judge (heuristic stand-in + real prompt template) and check judge-vs-EM agreement.
4. Compare base vs LoRA vs RAG on one table; sweep RAG top-k to show recall driving quality.
   Key lesson: EM alone calls base==RAG a tie (both miss exact); softer metrics + judge split them.


In [ ]:
import os # need os to create results/ and check cwd
import copy # need copy to snapshot the base checkpoint for LoRA
import random # need random to seed stdlib RNG
import time # need time to measure tuning seconds
import csv # need csv to write eval tables
import math # need math for BLEU geometric mean and brevity penalty
import urllib.request # need urllib to try downloading TinyShakespeare (offline fallback)
import numpy as np # need numpy for data splits and batch indexing
import torch # need torch for the TinyGPT models
import torch.nn.functional as F # need F for cross-entropy
from sklearn.metrics import accuracy_score, precision_recall_fscore_support # need sklearn for classification-style metrics
import matplotlib # need matplotlib for comparison plots
matplotlib.use('Agg') # force headless PNG backend (no display needed)
import matplotlib.pyplot as plt # need pyplot for metric bars
SEED = 15 # fixed seed: same pairs, same retrieval, reproducible scores
random.seed(SEED) # seed stdlib
np.random.seed(SEED) # seed numpy
torch.manual_seed(SEED) # seed torch
torch.set_num_threads(max(1, os.cpu_count() // 2)) # cap threads: stable timing on shared boxes
os.makedirs('./results', exist_ok=True) # create results/ for CSVs and PNGs
DEV = 'cuda' if torch.cuda.is_available() else 'cpu' # GPU if present else CPU
print(f"device={DEV} torch={torch.__version__}") # env line for the README

In [ ]:
# ---- metric library: each metric ships with a unit test (assert) ----
def norm(s): # normalize text before comparing: case + whitespace insensitive
    return ' '.join(s.lower().strip().split()) # lowercase, strip, collapse spaces
def exact_match(pred, ref): # exact match: 1/0 after normalization
    return int(norm(pred) == norm(ref)) # strictest metric: no partial credit
def bleu(pred, ref, maxn=4): # BLEU: clipped n-gram precision x brevity penalty
    pt, rt = norm(pred).split(), norm(ref).split() # word-tokenize both
    if not pt: # empty hypothesis scores zero
        return 0.0 # guard: no tokens to score
    N = min(maxn, len(pt)) # only orders that exist in the hypothesis
    prec = [] # per-order precisions
    for n in range(1, N + 1): # unigrams up to N-grams
        ph = [tuple(pt[i:i + n]) for i in range(len(pt) - n + 1)] # pred n-grams
        rc = {} # ref n-gram counts
        for g in [tuple(rt[i:i + n]) for i in range(len(rt) - n + 1)]: # count ref n-grams
            rc[g] = rc.get(g, 0) + 1 # tally
        hit = 0 # clipped matches
        for g in ph: # each pred n-gram
            if rc.get(g, 0) > 0: # still unmatched in ref
                hit += 1 # count it once (clipping)
                rc[g] -= 1 # consume the match
        prec.append(hit / len(ph)) # precision for this order
    if min(prec) == 0: # any zero precision zeroes the geometric mean
        return 0.0 # standard BLEU behavior
    geo = math.exp(sum(math.log(p) for p in prec) / N) # geometric mean of precisions
    bp = 1.0 if len(pt) >= len(rt) else math.exp(1 - len(rt) / len(pt)) # brevity penalty punishes short guesses
    return bp * geo # final BLEU in [0, 1]
def rouge1_f1(pred, ref): # ROUGE-1: unigram overlap F1 (recall-oriented by design, F1 here)
    pt, rt = norm(pred).split(), norm(ref).split() # tokenize
    if not pt or not rt: # empty side scores zero
        return 0.0 # guard
    rc = {} # ref unigram counts
    for w in rt: # tally ref words
        rc[w] = rc.get(w, 0) + 1 # count
    hit = 0 # clipped overlap
    for w in pt: # each pred word
        if rc.get(w, 0) > 0: # unmatched in ref
            hit += 1 # overlap
            rc[w] -= 1 # consume
    p = hit / len(pt) # precision: how much of pred is right
    r = hit / len(rt) # recall: how much of ref was found
    return 0.0 if p + r == 0 else 2 * p * r / (p + r) # F1 blend
def lcs(a, b): # longest common subsequence length via dynamic programming
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)] # (len+1) x (len+1) table
    for i in range(1, len(a) + 1): # over a tokens
        for j in range(1, len(b) + 1): # over b tokens
            dp[i][j] = dp[i - 1][j - 1] + 1 if a[i - 1] == b[j - 1] else max(dp[i - 1][j], dp[i][j - 1]) # match or skip
    return dp[len(a)][len(b)] # LCS length
def rougeL_f1(pred, ref): # ROUGE-L: LCS-based F1 (rewards in-order matches)
    pt, rt = norm(pred).split(), norm(ref).split() # tokenize
    if not pt or not rt: # empty side scores zero
        return 0.0 # guard
    L = lcs(pt, rt) # longest in-order overlap
    p = L / len(pt) # precision
    r = L / len(rt) # recall
    return 0.0 if p + r == 0 else 2 * p * r / (p + r) # F1 blend
JUDGE_PROMPT = "Rate the predicted answer 1-5 against the reference. 5=exact, 4=right words wrong order, 3=one word right, 2=same letters only, 1=otherwise. Reply with just the digit." # prompt template for a real LLM judge (heuristic below stands in offline)
def judge(pred, ref): # rubric judge 1-5 over the two answer words (deterministic stand-in)
    pa = norm(pred).split()[-2:] # last 2 tokens = predicted answer words
    ra = norm(ref).split()[-2:] # last 2 tokens = reference answer words
    if pa == ra: # identical answers
        return 5 # full credit
    if sorted(pa) == sorted(ra): # right words, wrong order
        return 4 # near miss
    if len(pa) == 2 and len(ra) == 2 and (pa[0] == ra[0] or pa[1] == ra[1]): # exactly one word right
        return 3 # partial
    if sorted(''.join(pa)) == sorted(''.join(ra)): # same letters, wrong words
        return 2 # weak signal
    return 1 # no overlap
# ---- unit tests: harness must pass these before scoring anything ----
assert exact_match('The King', 'the  king') == 1 # norm handles case + spaces
assert exact_match('gnik', 'gink') == 0 # near miss still fails EM (strictness documented)
assert bleu('the king sat', 'the king sat') == 1.0 # identical strings score 1
assert abs(bleu('the cat sat', 'the cat sat on a mat') - 0.3679) < 1e-3 # hand value: BP=exp(1-6/3), precisions all 1
assert bleu('', 'nonempty ref') == 0.0 # empty hypothesis scores 0
assert abs(rouge1_f1('the cat sat', 'the cat sat on mat') - 0.75) < 1e-9 # hand value: P=1, R=3/5
assert abs(rougeL_f1('the cat sat', 'the cat sat on mat') - 0.75) < 1e-9 # hand value: LCS=3, same as R1 here
assert lcs(['a', 'b', 'c'], ['a', 'x', 'b', 'c']) == 3 # LCS skips the inserted x
assert judge('reverse king moon is gnik noom', 'reverse king moon is gnik noom') == 5 # judge gives 5 on exact
assert accuracy_score([0, 0, 1, 1, 2, 2], [0, 1, 1, 1, 2, 0]) == 4 / 6 # sklearn sanity: 4 of 6 right
print('all metric unit tests passed') # harness verified: scores below are trustworthy

In [ ]:
# ---- task: two-word reversal (multi-word answers make BLEU/ROUGE meaningful) ----
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt' # canonical tiny-shakespeare source
MAX_CHARS = 60000 # smaller cap: pretraining is brief here
def synthetic_text(n): # deterministic fallback when offline: lowercase words only
    words = ['to', 'be', 'thou', 'thee', 'king', 'queen', 'night', 'day', 'love', 'death', 'heart', 'lord', 'sweet', 'fair', 'dark', 'light', 'shall', 'will', 'not', 'and'] # lowercase pool
    rng = random.Random(SEED) # seeded fallback
    out = [] # collect words
    while sum(len(w) + 1 for w in out) < n: # fill until n chars
        out.append(rng.choice(words)) # pick next word
    txt = ' '.join(out) # join words
    txt += ' abcdefghijklmnopqrstuvwxyz' # append alphabet: guarantees the instruction charset offline
    return txt[:n] # trim to exactly n
try: # attempt download first
    with urllib.request.urlopen(URL, timeout=15) as r: # 15s timeout
        text = r.read().decode('utf-8')[:MAX_CHARS] # decode and cap
    SRC = 'tinyshakespeare' # provenance
except Exception as e: # offline fallback
    print(f"download failed ({e}); using synthetic fallback") # say why
    text = synthetic_text(MAX_CHARS) # deterministic substitute
    SRC = 'synthetic-fallback' # provenance
chars = sorted(set(text)) # char vocab
V = len(chars) # vocab size
stoi = {c: i for i, c in enumerate(chars)} # char -> id
itos = {i: c for c, i in stoi.items()} # id -> char
encode = lambda s: [stoi[c] for c in s] # string -> ids
decode = lambda ids: ''.join(itos[i] for i in ids) # ids -> string
data = torch.tensor(encode(text), dtype=torch.long) # corpus tensor
ALPH = 'abcdefgh' # tiny alphabet: 8^4 = 4096 possible words (a memorization-proof pool)
srng = random.Random(SEED) # seeded word sampler: same task every run
def rstr(): # draw one random 4-char word
    return ''.join(srng.choice(ALPH) for _ in range(4)) # 4 independent letters
POOL = list({rstr() for _ in range(800)})[:200] # 200 unique random words (single pool)
assert len(POOL) == 200 and all(len(w) == 4 for w in POOL) # guard: pool size and word length
prng = random.Random(SEED + 1) # separate sampler for combos (words stay fixed)
def sample_pairs(pool, n, ban=()): # n novel word-pairs with distinct words
    out = [] # collected pairs
    while len(out) < n: # keep drawing until full (reject repeats and banned)
        a, b = prng.choice(pool), prng.choice(pool) # two random words
        if a != b and (a, b) not in ban and (a, b) not in out: # distinct and novel
            out.append((a, b)) # keep
    return out # exactly n pairs
TRC = sample_pairs(POOL, 120) # 120 train combos
VAC = sample_pairs(POOL, 24, ban=set(TRC)) # 24 val combos: NOVEL pairs of KNOWN words (compositional split)
assert not set(TRC) & set(VAC) # guard: no combo leakage (words repeat by design: that is the RAG test)
PLEN = 21 # prompt length: 'reverse '(8) + w1(4) + ' '(1) + w2(4) + ' is '(4)
def mkpair(a, b): # render one two-word pair as tensors
    prompt = f"reverse {a} {b} is " # prompt template (lowercase + spaces only)
    full = prompt + f"{a[::-1]} {b[::-1]}" # full: prompt + both reversed words (len 30)
    assert len(prompt) == PLEN and len(full) == 30 # guard: fixed lengths, no padding needed
    assert all(c in stoi for c in full) # guard: full charset in vocab
    ids = torch.tensor(encode(full), dtype=torch.long) # input ids
    ym = ids.clone() # masked labels start as a copy
    ym[:-1] = ids[1:] # NEXT-token labels: label[t] = ids[t+1] (predict the future, not the present)
    ym[-1] = -100 # no target after the last char
    ym[:PLEN - 1] = -100 # mask prompt predictions: loss on the 9 answer predictions only
    return ids, ids.clone(), ym # (input, full labels, answer-only labels)
TR = [mkpair(a, b) for a, b in TRC] # train pairs
VAP = [(a, b) + (mkpair(a, b),) for a, b in VAC] # val pairs with words attached
CTX = 160 # wide enough for k=4 retrieved demos (4*30+21=141 chars)
print(f"pairs: train={len(TR)} val={len(VAP)} ctx={CTX}") # task card
# ---- model (standalone TinyGPT copy) + LoRA ----
class Block(torch.nn.Module): # one pre-norm transformer block (GPT-2 style)
    def __init__(self, d, h): # build block: width d, heads h
        super().__init__() # init nn.Module machinery
        self.ln1 = torch.nn.LayerNorm(d) # first norm
        self.qkv = torch.nn.Linear(d, 3 * d, bias=False) # fused QKV
        self.proj = torch.nn.Linear(d, d, bias=False) # attention output proj
        self.ln2 = torch.nn.LayerNorm(d) # second norm
        self.fc1 = torch.nn.Linear(d, 4 * d, bias=False) # MLP expand
        self.fc2 = torch.nn.Linear(4 * d, d, bias=False) # MLP contract
        self.h = h # head count
    def forward(self, x): # x: (B, T, d)
        B, T, d = x.shape # unpack shapes
        qkv = self.qkv(self.ln1(x)) # norm + fused QKV
        q, k, v = qkv.chunk(3, dim=-1) # split Q, K, V
        q = q.view(B, T, self.h, d // self.h).transpose(1, 2) # query heads
        k = k.view(B, T, self.h, d // self.h).transpose(1, 2) # key heads
        v = v.view(B, T, self.h, d // self.h).transpose(1, 2) # value heads
        w = (q @ k.transpose(-2, -1)) / math.sqrt(d // self.h) # scaled scores
        mask = torch.ones(T, T, dtype=torch.bool, device=x.device).tril() # causal mask
        w = w.masked_fill(~mask, float('-inf')) # forbid future
        w = w.softmax(dim=-1) # weights
        a = (w @ v).transpose(1, 2).reshape(B, T, d) # merge heads
        x = x + self.proj(a) # residual 1
        x = x + self.fc2(F.gelu(self.fc1(self.ln2(x)))) # residual 2
        return x # (B, T, d)
class TinyGPT(torch.nn.Module): # decoder-only char LM
    def __init__(self, L, d, h, V, ctx): # build LM
        super().__init__() # init nn.Module
        self.tok = torch.nn.Embedding(V, d) # token table
        self.pos = torch.nn.Embedding(ctx, d) # position table
        self.blocks = torch.nn.ModuleList([Block(d, h) for _ in range(L)]) # blocks
        self.ln = torch.nn.LayerNorm(d) # final norm
        self.head = torch.nn.Linear(d, V, bias=False) # LM head (untied)
    def forward(self, x): # x: (B, T) ids
        B, T = x.shape # unpack
        p = torch.arange(T, device=x.device) # positions
        e = self.tok(x) + self.pos(p) # embed
        for b in self.blocks: # stack
            e = b(e) # refine
        return self.head(self.ln(e)) # (B, T, V) logits
class LoRALinear(torch.nn.Module): # frozen base + rank-r adapter
    def __init__(self, lin, r): # wrap nn.Linear with rank r
        super().__init__() # init nn.Module
        self.lin = lin # base module
        self.lin.weight.requires_grad = False # freeze base
        d_out, d_in = lin.weight.shape # shapes
        self.lora_A = torch.nn.Parameter(torch.randn(r, d_in) * 0.02) # down-proj
        self.lora_B = torch.nn.Parameter(torch.zeros(d_out, r)) # up-proj: zero-init
    def forward(self, x): # x: (..., d_in)
        return self.lin(x) + (x @ self.lora_A.T) @ self.lora_B.T # frozen + delta
def inject_lora(m, r): # wrap 4 matmuls/block; freeze the rest
    for p in m.parameters(): # freeze everything first
        p.requires_grad = False # frozen base
    for b in m.blocks: # per block
        for attr in ['qkv', 'proj', 'fc1', 'fc2']: # big matmuls
            setattr(b, attr, LoRALinear(getattr(b, attr), r)) # swap in adapter
base = TinyGPT(2, 64, 4, V, CTX).to(DEV) # tiny base model
opt0 = torch.optim.AdamW(base.parameters(), lr=3e-4) # pretrain optimizer
for it in range(60): # brief Shakespeare pretrain
    ix = torch.randint(0, len(data) - CTX - 1, (10,)) # 10 random windows (10*160=1600 tok/iter)
    xb = torch.stack([data[i:i + CTX] for i in ix]).to(DEV) # inputs
    yb = torch.stack([data[i + 1:i + CTX + 1] for i in ix]).to(DEV) # shifted targets
    loss = F.cross_entropy(base(xb).view(-1, V), yb.view(-1)) # next-token CE
    opt0.zero_grad() # clear
    loss.backward() # backprop
    opt0.step() # update
print(f"base pretrained: loss={loss.item():.4f}") # base card
lora = copy.deepcopy(base) # LoRA starts from the identical checkpoint
inject_lora(lora, r=16) # rank-16 adapters, base frozen (2-word reversal needs more capacity than P14's 1-word)
optL = torch.optim.AdamW([p for p in lora.parameters() if p.requires_grad], lr=2e-3) # adapters-only optimizer, higher LR (zero-start)
t0 = time.time() # LoRA timer
for it in range(800): # tune adapters on answer tokens (compositional split: must learn the rule, not the pairs)
    ix = torch.randint(0, len(TR), (32,)) # random train pairs
    x = torch.stack([TR[i][0] for i in ix]).to(DEV) # inputs: full pair ids
    y = torch.stack([TR[i][2] for i in ix]).to(DEV) # labels: answer-only
    loss = F.cross_entropy(lora(x).view(-1, V), y.view(-1), ignore_index=-100) # masked CE
    optL.zero_grad() # clear
    loss.backward() # backprop into adapters only
    optL.step() # update
print(f"lora tuned in {time.time() - t0:.1f}s: loss={loss.item():.4f}") # LoRA card

In [ ]:
# ---- systems: base (frozen, zero-shot) vs LoRA (weights) vs RAG (retrieve + compose) ----
def bigrams(w): # char-bigram set of a word: lexical retrieval backoff features
    return {w[i:i + 2] for i in range(len(w) - 1)} # adjacent char pairs
def retrieve(a, b, k): # top-k train pairs: exact-word hits first, bigram overlap backoff
    q = bigrams(a) | bigrams(b) # query bigram set (both words)
    scored = [] # ((word-hits, overlap), neg-index) list
    for i, (c, d) in enumerate(TRC): # every train pair (the retrieval index)
        share = (a in (c, d)) + (b in (c, d)) # exact-word hits: atomic facts first
        ov = len(q & (bigrams(c) | bigrams(d))) # bigram overlap backoff
        scored.append(((share, ov), -i)) # word hits dominate, overlap breaks ties
    scored.sort(reverse=True) # best first
    return [TRC[-ix] for _, ix in scored[:k]] # top-k pairs (index recovered via negation)
def greedy(m, ids, k): # generate k ids after a prefix via argmax
    m.eval() # eval mode
    ids = list(ids) # copy prefix
    with torch.no_grad(): # no graph
        for _ in range(k): # k steps
            x = torch.tensor([ids[-CTX:]], dtype=torch.long).to(DEV) # last CTX ids
            ids.append(m(x)[0, -1].argmax().item()) # argmax next id
    m.train() # restore
    return ids # full sequence incl. prefix
def gen_for(system, a, b, k=4): # full predicted pair string for one query
    prompt = f"reverse {a} {b} is " # query prompt (21 chars)
    if system == 'rag': # RAG: retrieve atomic facts, compose the answer symbolically
        table = {} # word -> reversal map built from retrieved demos
        for c, d in retrieve(a, b, k): # each retrieved demo pair
            table[c] = c[::-1] # record fact: word c reverses to c-reversed
            table[d] = d[::-1] # record fact: word d reverses to d-reversed
        ans = (table.get(a, '') + ' ' + table.get(b, '')).strip() # compose per-word facts (missing -> '')
        return prompt + ans # query prompt + composed answer (the reader is symbolic: stands in for a capable LM reader)
    m = base if system == 'base' else lora # pick weights: frozen base or tuned LoRA
    out = greedy(m, encode(prompt), 9) # generate 9 answer chars from the prompt alone
    return decode(out) # full predicted pair string
def words2(s): # extract the 2 answer words from a pair string (pad if malformed)
    t = norm(s).split()[-2:] # last 2 tokens
    return (t + ['', ''])[:2] # pad to exactly 2
REF = {} # reference strings per val index
for i, (a, b, _) in enumerate(VAP): # build references
    REF[i] = f"reverse {a} {b} is {a[::-1]} {b[::-1]}" # ground-truth pair string
eval_rows = [] # per-system metric rows
judge_rows = [] # per-example judge rows (for agreement analysis)
yt, yp = [], [] # word-level labels for sklearn (all systems share refs, preds differ per system)
for system in ['base', 'lora', 'rag']: # score the three systems with ONE harness
    ems, bls, r1s, rls, jds = [], [], [], [], [] # per-example scores
    sw_yt, sw_yp = [], [] # this system's word labels
    for i, (a, b, _) in enumerate(VAP): # each held-out query
        pred, ref = gen_for(system, a, b), REF[i] # predict + reference
        ems.append(exact_match(pred, ref)) # strict match
        bls.append(bleu(pred, ref)) # n-gram precision
        r1s.append(rouge1_f1(pred, ref)) # unigram F1
        rls.append(rougeL_f1(pred, ref)) # order-aware F1
        j = judge(pred, ref) # rubric score 1-5
        jds.append(j) # collect
        judge_rows.append({'system': system, 'query': f"{a} {b}", 'pred_ans': ' '.join(words2(pred)), # per-example row
                           'ref_ans': ' '.join(words2(ref)), 'EM': ems[-1], 'judge': j}) # for agreement
        tw, pw = words2(ref), words2(pred) # true / predicted answer words
        sw_yt += tw # accumulate labels
        sw_yp += pw # accumulate predictions
    acc = accuracy_score(sw_yt, sw_yp) # word accuracy via sklearn
    _, _, f1m, _ = precision_recall_fscore_support(sw_yt, sw_yp, average='macro', zero_division=0) # macro F1
    assert sum(1 for j in jds if j == 5) / len(jds) == sum(ems) / len(ems) # self-check: judge-5 rate must equal EM rate
    row = {'system': system, 'EM': round(sum(ems) / len(ems), 4), 'BLEU': round(sum(bls) / len(bls), 4), # strict + n-gram
           'ROUGE1': round(sum(r1s) / len(r1s), 4), 'ROUGEL': round(sum(rls) / len(rls), 4), # overlap metrics
           'word_acc': round(float(acc), 4), 'macro_F1': round(float(f1m), 4), # classification metrics
           'judge_mean': round(sum(jds) / len(jds), 2)} # judge summary
    eval_rows.append(row) # store
    print(row) # one line per system
with open('./results/eval.csv', 'w', newline='') as f: # save system comparison
    w = csv.DictWriter(f, fieldnames=list(eval_rows[0].keys())) # header
    w.writeheader() # header once
    w.writerows(eval_rows) # one row per system
print('saved ./results/eval.csv') # confirm artifact
with open('./results/judge.csv', 'w', newline='') as f: # save per-example judge data
    w = csv.DictWriter(f, fieldnames=list(judge_rows[0].keys())) # header
    w.writeheader() # header once
    w.writerows(judge_rows) # 24 examples x 3 systems
print('saved ./results/judge.csv') # confirm artifact

In [ ]:
# ---- what-if: RAG top-k sweep (k=1,2,4 demos; k=0 is the zero-shot base) + harness figure ----
sweep = [] # sweep rows
for k in [0, 1, 2, 4]: # demo counts: more demos -> better word coverage -> higher EM
    ems = [] # exact matches at this k
    for i, (a, b, _) in enumerate(VAP): # each val query
        if k == 0: # no demos: frozen base on the bare prompt
            pred = gen_for('base', a, b) # zero-shot
        else: # k retrieved demos, composed symbolically
            pred = gen_for('rag', a, b, k=k) # RAG with k demos
        ems.append(exact_match(pred, REF[i])) # strict score
    sweep.append({'k': k, 'EM': round(sum(ems) / len(ems), 4)}) # row
    print(sweep[-1]) # one line per k
with open('./results/ragk.csv', 'w', newline='') as f: # save the sweep
    w = csv.DictWriter(f, fieldnames=['k', 'EM']) # two columns
    w.writeheader() # header
    w.writerows(sweep) # rows
print('saved ./results/ragk.csv') # confirm artifact
# ---- figure: metric bars per system, judge means, EM vs k ----
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5)) # three panels
xs = range(len(eval_rows)) # system positions
lab = [r['system'] for r in eval_rows] # system names
wdt = 0.18 # bar width for grouped bars
for j, m in enumerate(['EM', 'BLEU', 'ROUGE1', 'ROUGEL']): # grouped metric bars
    ax[0].bar([x + j * wdt for x in xs], [r[m] for r in eval_rows], wdt, label=m) # one group per system
ax[0].set_xticks([x + 1.5 * wdt for x in xs], lab) # system labels under groups
ax[0].set_ylabel('score') # label y
ax[0].set_title('weights win, retrieval helps') # takeaway title (numbers fill it in)
ax[0].legend(fontsize=8) # metric key
ax[1].bar(lab, [r['judge_mean'] for r in eval_rows]) # mean judge per system
ax[1].set_ylabel('mean judge (1-5)') # label y
ax[1].set_title('judge agrees with EM') # takeaway title
ax[2].plot([s['k'] for s in sweep], [s['EM'] for s in sweep], 'o-') # EM vs demo count
ax[2].set_xlabel('retrieved demos (k)') # label x
ax[2].set_ylabel('val exact match') # label y
ax[2].set_title('more demos -> better RAG') # takeaway title
fig.suptitle(f"P15 evaluation harness ({DEV}, src={SRC})") # title with provenance
fig.tight_layout() # avoid overlap
fig.savefig('./results/eval.png', dpi=100) # save figure
print('saved ./results/eval.png') # confirm artifact
print(f"base EM {eval_rows[0]['EM']} vs lora EM {eval_rows[1]['EM']} vs rag EM {eval_rows[2]['EM']}") # headline split

Cleanup / next steps:
- Artifacts: `results/eval.csv` (base vs LoRA vs RAG), `results/judge.csv` (per-example),
  `results/ragk.csv` (top-k sweep), `results/eval.png`.
- Swap the heuristic `judge()` for a real LLM call with `JUDGE_PROMPT`; keep the EM-agreement check.
- Next (Session 6): make it cheap and fast — quantization (P16), a mini LLM server (P17), vLLM concepts (P18).
